In [14]:
from dotenv import load_dotenv
load_dotenv()

from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough, RunnableParallel
from langchain_google_genai import ChatGoogleGenerativeAI

In [15]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
)

In [16]:
animal_facts_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You like telling facts and you tell facts about {animal}."),
        ("human", "Tell me {count} facts."),
    ]
)

translation_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a translator and convert the provided text into {language}."),
        ("human", "Translate the following text to {language}: {english_text}"),
    ]
)

In [21]:
chain = (
    RunnableParallel({
        "english_text": animal_facts_template | llm | StrOutputParser(),
        "language": lambda x: x['language'],
    })
    | RunnableParallel({
        "english_text": lambda x: x['english_text'],
        "translated_text": translation_template | llm | StrOutputParser(),
        "language": lambda x: x['language'],
    })
)

In [24]:
response = chain.invoke({
    "animal": "cats",
    "count": 1,
    "language": "hindi"
})

# print("Response:", response)
print(f"english: {response['english_text']}")
print(f"{response['language']} translation: {response['translated_text']}")

english: Did you know that a group of cats is called a clowder?
hindi translation: क्या आपको पता है कि बिल्लियों के एक समूह को 'क्लाउडर' (clowder) कहा जाता है?
